In [53]:
# Get Data
concurrent_requests = 64
batch_size = 1  # Reduced for better progress visibility
timeout_seconds = 120
use_cookies = False
connection_retry_delay = 30
max_connection_retries = 3
db_loc = '../../data/driversf.db'

import logging
import asyncio
import aiohttp
import json
import random
import pandas as pd
import duckdb
from sqlalchemy.exc import ProgrammingError
from sqlalchemy import text, inspect, create_engine
from tqdm.notebook import tqdm
import nest_asyncio
import time
from concurrent.futures import ThreadPoolExecutor

    
parts = ['mostReplayed', 'chapters', 'activity'] # 'snippet', 'statistics', 'contentDetails'
part_str = ','.join(parts)

# Configure TCP connector
connector = aiohttp.TCPConnector(
    limit=concurrent_requests,
    force_close=True,
    enable_cleanup_closed=True,
    ttl_dns_cache=300
)

semaphore = asyncio.Semaphore(concurrent_requests)

# Create a thread pool for database operations
db_pool = ThreadPoolExecutor(max_workers=4)

async def exponential_backoff(attempt, max_delay=60):
    delay = min(2 ** attempt + random.uniform(0, 1), max_delay)
    await asyncio.sleep(delay)

async def fetch_data(session, video_ids_batch, max_retries=5):
    """Fetch data for multiple video IDs in a single request"""
    async with semaphore:
        url = f'http://localhost:8080/videos?part={part_str}&id={video_ids_batch}'
        
        for attempt in range(max_retries):
            try:
                async with session.get(url, timeout=aiohttp.ClientTimeout(total=timeout_seconds)) as response:
                    if response.status == 429:
                        await exponential_backoff(attempt)
                        continue
                    
                    response.raise_for_status()
                    text_response = await response.text()
                    
                    json_start = text_response.find('{')
                    if json_start != -1:
                        try:
                            json_data = json.loads(text_response[json_start:])
                            return json_data['items']
                        except json.JSONDecodeError as e:
                            logging.error(f"JSON decode error: {e}")
                            await exponential_backoff(attempt)
                    return None
            
            except (aiohttp.ClientConnectorError, 
                   aiohttp.ServerDisconnectedError, 
                   aiohttp.ClientResponseError) as e:
                if "10053" in str(e):
                    logging.warning(f"Connection aborted (Error 10053). Waiting...")
                    await asyncio.sleep(connection_retry_delay)
                else:
                    logging.error(f"Connection error: {type(e).__name__}: {str(e)}")
                    await exponential_backoff(attempt)
            except Exception as e:
                logging.error(f"Unexpected error: {type(e).__name__}: {str(e)}")
                await exponential_backoff(attempt)

        with open('failed_video_id_batches.txt', 'a') as f:
            f.write(video_ids_batch + '\n')
        return None

def db_insert(df, engine):
    """Execute database insert in a separate thread"""
    try:
        df.to_sql('scraped', con=engine, if_exists='append', index=True)
        return True
    except ProgrammingError as e:
        if "does not have" in str(e).lower():
            add_missing_columns(df, engine)
            df.to_sql('scraped', con=engine, if_exists='append', index=True)
            return True
        logging.error(f"Database error: {e}")
        return False
    except Exception as e:
        logging.error(f"Database insert error: {e}")
        return False

async def fetch_all_data(video_ids):
    """Process all video IDs with progress tracking"""
    data = []
    engine = create_engine(f'duckdb:///{db_loc}')
    
    # Create batches
    batches = [','.join(video_ids[i:i + batch_size]) 
               for i in range(0, len(video_ids), batch_size)]
    
    session_args = {
        'connector': connector,
        'timeout': aiohttp.ClientTimeout(total=timeout_seconds),
        'trust_env': True
    }

    if use_cookies:
        session_args['cookies'] = cookies

    async with aiohttp.ClientSession(**session_args) as session:
        pbar = tqdm(total=len(batches), desc="Processing video batches")
        tasks = set()
        results = []
        
        # Process batches with controlled concurrency
        batch_queue = batches.copy()
        while batch_queue or tasks:
            # Fill up tasks to concurrent_requests limit
            while len(tasks) < concurrent_requests and batch_queue:
                batch = batch_queue.pop(0)
                task = asyncio.create_task(fetch_data(session, batch))
                tasks.add(task)
                
            if not tasks:
                break
                
            # Wait for any task to complete
            done, tasks = await asyncio.wait(
                tasks, 
                return_when=asyncio.FIRST_COMPLETED
            )
            
            # Process completed tasks
            for task in done:
                try:
                    result = await task
                    if result:
                        df = treat_result(result)
                        # Execute DB insert in thread pool
                        loop = asyncio.get_event_loop()
                        success = await loop.run_in_executor(
                            db_pool, db_insert, df, engine
                        )
                        if success:
                            results.append(df)
                except Exception as e:
                    logging.error(f"Task processing error: {e}")
                finally:
                    pbar.update(1)
        
        pbar.close()
        return results

def treat_result(result):
    treated_result = pd.json_normalize(result)
    if 'id' in treated_result.columns:
        treated_result.set_index('id', inplace=True)
    for col in treated_result.columns:
        treated_result[col] = treated_result[col].apply(json.dumps)
    return treated_result

def add_missing_columns(result_df, engine):
    inspector = inspect(engine)
    existing_columns = [col['name'] for col in inspector.get_columns('scraped')]
    new_columns = set(result_df.columns) - set(existing_columns)
    
    if new_columns:
        with engine.begin() as conn:
            for col in new_columns:
                alter_table_query = f'ALTER TABLE scraped ADD COLUMN "{col}" TEXT'
                conn.execute(text(alter_table_query))

async def main():
    iteration = 0
    while True:
        try:
            logging.info(f"Starting iteration {iteration}")
            
            # Get video IDs
            query = "SELECT DISTINCT id FROM '../../data/driver_san_francisco_video_statistics.parquet'"
            all_video_ids = set(duckdb.query(query).df()['id'].unique())
            
            # Get already scraped IDs
            try:
                with duckdb.connect(db_loc) as conn:
                    scraped_ids = set(conn.execute("SELECT DISTINCT id FROM scraped").df()['id'].unique())
                remaining_ids = list(all_video_ids - scraped_ids)
                logging.info(f"Found {len(scraped_ids)} already scraped videos")
            except Exception:
                remaining_ids = list(all_video_ids)
                logging.info("No existing 'scraped' table found")
            
            if not remaining_ids:
                logging.info("No new videos to scrape. Exiting.")
                break
            
            logging.info(f"Fetching data for {len(remaining_ids)} videos")
            
            start = time.time()
            results = await fetch_all_data(remaining_ids)
            end = time.time()
            
            processed_count = len(results)
            logging.info(f"Processed {processed_count} videos in {end - start:.2f} seconds")
            
            if processed_count == 0:
                logging.warning("No videos were processed successfully in this iteration")
                await asyncio.sleep(connection_retry_delay)
            
            iteration += 1
            
        except Exception as e:
            logging.critical(f"Critical error in main loop: {e}")
            await asyncio.sleep(connection_retry_delay)

if __name__ == "__main__":
    nest_asyncio.apply()
    asyncio.run(main())

Processing video batches:   0%|          | 0/8100 [00:00<?, ?it/s]

ERROR:root:Unexpected error: TimeoutError: 
ERROR:root:Unexpected error: TimeoutError: 
ERROR:root:Unexpected error: TimeoutError: 
ERROR:root:Unexpected error: TimeoutError: 
ERROR:root:Unexpected error: TimeoutError: 
ERROR:root:Unexpected error: TimeoutError: 
ERROR:root:Unexpected error: TimeoutError: 
ERROR:root:Unexpected error: TimeoutError: 
ERROR:root:Unexpected error: TimeoutError: 
ERROR:root:Unexpected error: TimeoutError: 
ERROR:root:Unexpected error: TimeoutError: 
ERROR:root:Unexpected error: TimeoutError: 


In [ ]:
# Write to parquet

import duckdb

query = """
        SELECT  
            "id",
            "activity.name",
        FROM scraped
        """

print('Connecting to read_games...')
conn = duckdb.connect('../../data/read_games.db')

print('Executing query...')
read_games_data = conn.execute(query)

print('Fetching...')
read_games_df = read_games_data.pl()

print('Closing...')
conn.close()

print('Connecting to games...')
conn = duckdb.connect('../../data/games.db')

print('Executing query...')
games_data = conn.execute(query)

print('Fetching...')
games_df = games_data.pl()

print('Closing...')
conn.close()

import polars as pl

print('Concatenating...')
df = pl.concat([read_games_df, games_df])

print('Removing nulls...')
df = df.filter(pl.col('activity.name') != 'null')

print('Dropping duplicates...')
df = df.unique(subset=['id'])

print('Writing...')
df.write_parquet('../../data/games.parquet')

print('Reading...')
df = pl.read_parquet('../../data/games.parquet')

df.shape

In [54]:
import polars as pl
import duckdb

conn = duckdb.connect('../../data/driversf.db')
scraped_driversf_df =  conn.execute("SELECT * FROM scraped").pl()
scraped_driversf_df = scraped_driversf_df.filter(pl.col('activity.name').str.contains('Driver: San Francisco'))

driversf_df = duckdb.query("SELECT * FROM '../../data/driver_san_francisco_video_statistics.parquet'").pl()

driversf_df = driversf_df.join(scraped_driversf_df, on='id', how='inner')

languages = ['en', 'en-US', 'en-GB', 'en-IN', 'zxx', 'en-CA', 'en-AU', 'null', None]

driversf_df = driversf_df.filter(
    pl.col('contentDetails.dimension') == '2d',
    pl.col('contentDetails.definition') == 'hd',
    pl.col('contentDetails.projection') == 'rectangular',
    pl.col('snippet.categoryId') == '20',
    pl.col('snippet.liveBroadcastContent') == 'none',
    pl.col('snippet.defaultAudioLanguage').is_in(languages) | pl.col('snippet.defaultAudioLanguage').is_null(),
    pl.col('snippet.defaultLanguage').is_in(languages) | pl.col('snippet.defaultLanguage').is_null(),
    pl.col('liveStreamingDetails.actualStartTime').is_null(),
    pl.col('liveStreamingDetails.actualEndTime').is_null(),
    pl.col('liveStreamingDetails.scheduledStartTime').is_null(),
)

driversf_df = driversf_df.drop([col for col in driversf_df.columns if 'favoriteCount' in col])
driversf_df = driversf_df.drop([col for col in driversf_df.columns if 'dimension' in col])
driversf_df = driversf_df.drop([col for col in driversf_df.columns if 'caption' in col])
driversf_df = driversf_df.drop([col for col in driversf_df.columns if 'projection' in col])
driversf_df = driversf_df.drop([col for col in driversf_df.columns if 'definition' in col])
driversf_df = driversf_df.drop([col for col in driversf_df.columns if 'categoryId' in col])
driversf_df = driversf_df.drop([col for col in driversf_df.columns if 'liveBroadcastContent' in col])
driversf_df = driversf_df.drop([col for col in driversf_df.columns if 'thumbnail' in col])
driversf_df = driversf_df.drop([col for col in driversf_df.columns if 'etag' in col])
driversf_df = driversf_df.drop([col for col in driversf_df.columns if 'kind' in col])
driversf_df = driversf_df.drop([col for col in driversf_df.columns if 'localized' in col])
driversf_df = driversf_df.drop([col for col in driversf_df.columns if 'region' in col])
driversf_df = driversf_df.drop([col for col in driversf_df.columns if 'license' in col])
driversf_df = driversf_df.drop([col for col in driversf_df.columns if 'Language' in col])
driversf_df = driversf_df.drop([col for col in driversf_df.columns if 'actualStartTime' in col])
driversf_df = driversf_df.drop([col for col in driversf_df.columns if 'actualEndTime' in col])
driversf_df = driversf_df.drop([col for col in driversf_df.columns if 'scheduledStartTime' in col])

driversf_df = driversf_df.with_columns(
    pl.col('statistics.viewCount').cast(pl.Int32),
    pl.col('statistics.likeCount').cast(pl.Int32),
    pl.col('statistics.commentCount').cast(pl.Int32),
)

driversf_df.shape

(8445, 21)

In [55]:
driversf_df.sort('statistics.viewCount')

id,snippet.publishedAt,snippet.channelId,snippet.title,snippet.description,snippet.channelTitle,snippet.tags,contentDetails.duration,statistics.viewCount,statistics.likeCount,statistics.commentCount,topicDetails.topicCategories,liveStreamingDetails.scheduledEndTime,mostReplayed,chapters.areAutoGenerated,chapters.chapters,activity.name,activity.year,activity.channelId,mostReplayed.timedMarkerDecorations,mostReplayed.markers
str,str,str,str,str,str,list[str],str,i32,i32,i32,list[str],str,str,str,str,str,str,str,str,str
"""RI5xEN0YNoo""","""2015-02-15T20:33:56Z""","""UCCo4eB17KG1IkmvBSqFVcFQ""","""Driver San Francisco || Part 1…","""Thanks for watching! Please li…","""RebelYelliex""","[""rebelyelliex""]","""PT9M59S""",0,0,0,"[""https://en.wikipedia.org/wiki/Action-adventure_game"", ""https://en.wikipedia.org/wiki/Action_game"", … ""https://en.wikipedia.org/wiki/Video_game_culture""]",null,"""null""","""false""","""[]""","""""Driver: San Francisco""""","""""2011""""","""""UCuTo_0SBaeadIygt8JV2kcA""""",null,null
"""u0yIiq36Yvk""","""2021-04-27T13:49:02Z""","""UCQ77-4qHFc4HMxsKcE8E4WA""","""DRIVER: San Francisco - Activi…","""""","""ScarletMarisa375""",null,"""PT1M32S""",0,0,0,"[""https://en.wikipedia.org/wiki/Action-adventure_game"", ""https://en.wikipedia.org/wiki/Action_game"", … ""https://en.wikipedia.org/wiki/Video_game_culture""]",null,"""null""","""false""","""[]""","""""Driver: San Francisco""""","""""2011""""","""""UCuTo_0SBaeadIygt8JV2kcA""""",null,null
"""c0eucPYorVs""","""2019-07-15T21:38:52Z""","""UCvFq2r1ohx8PsQOGbfB0ybQ""","""Driver San Francisco #2""","""Broadcasted live on Twitch -- …","""CJRT Media""","[""twitch"", ""games""]","""PT28M2S""",0,0,0,"[""https://en.wikipedia.org/wiki/Action-adventure_game"", ""https://en.wikipedia.org/wiki/Action_game"", … ""https://en.wikipedia.org/wiki/Video_game_culture""]",null,"""null""","""false""","""[]""","""""Driver: San Francisco""""","""""2011""""","""""UCuTo_0SBaeadIygt8JV2kcA""""",null,null
"""tuaGUZ91G5U""","""2019-03-26T20:36:26Z""","""UC4Fn7axsJKvJp5pj3MMUN2A""","""Driver San Francisco Won by …","""If the video is long the inter…","""klisurovi4""",null,"""PT5M1S""",0,0,null,"[""https://en.wikipedia.org/wiki/Action_game"", ""https://en.wikipedia.org/wiki/Racing_video_game"", ""https://en.wikipedia.org/wiki/Video_game_culture""]",null,"""null""","""false""","""[]""","""""Driver: San Francisco""""","""""2011""""","""""UCuTo_0SBaeadIygt8JV2kcA""""",null,null
"""e4s82HCte4M""","""2024-05-19T19:28:18Z""","""UCzH4aePuBAgGt9jlefhFP-g""","""DRIVER San Francisco - New Ski…","""Lets get it! x""","""Lov3toPlay""","[""DRIVER San Francisco - New Skill""]","""PT50S""",0,0,0,"[""https://en.wikipedia.org/wiki/Action-adventure_game"", ""https://en.wikipedia.org/wiki/Action_game"", … ""https://en.wikipedia.org/wiki/Video_game_culture""]",null,"""null""","""false""","""[]""","""""Driver: San Francisco""""","""""2011""""","""""UCuTo_0SBaeadIygt8JV2kcA""""",null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""22qIFtavMmU""","""2022-05-12T12:29:08Z""","""UCHTyJ3httJ0pGXj4Vd9Mwbg""","""Evolution of DRIVER Games 1999…","""Includes Gameplay From ALL Dri…","""Game Evolutions""","[""evolution of driver games 1999-2021"", ""Driver Games"", … ""ubisoft""]","""PT8M55S""",812473,9472,568,"[""https://en.wikipedia.org/wiki/Action-adventure_game"", ""https://en.wikipedia.org/wiki/Action_game"", … ""https://en.wikipedia.org/wiki/Video_game_culture""]",null,null,"""false""","""[{""title"": ""Driver: You Are th…","""""Driver: San Francisco""""","""""2011""""","""""UCuTo_0SBaeadIygt8JV2kcA""""","""[{""visibleTimeRangeStartMillis…","""[{""startMillis"": 0, ""intensity…"
"""22qIFtavMmU""","""2022-05-12T12:29:08Z""","""UCHTyJ3httJ0pGXj4Vd9Mwbg""","""Evolution of DRIVER Games 1999…","""Includes Gameplay From ALL Dri…","""Game Evolutions""","[""evolution of driver games 1999-2021"", ""Driver Games"", … ""ubisoft""]","""PT8M55S""",812473,9472,568,"[""https://en.wikipedia.org/wiki/Action-adventure_game"", ""https